# 13 — Sınıflandırma modellerinin ROC-AUC ile yeniden değerlendirilmesi (Kaggle)

**Ayarlar:** Accelerator **GPU T4 x2** · Internet **On** · Secrets: `GH_TOKEN`. Sonra **Save Version → Save & Run All (Commit)**.

Hocanın teslim kurallarında sınıflandırma için **ROC-AUC** isteniyor; ilk eğitimlerde sınıf olasılıkları kaydedilmediği için hesaplanamamıştı. Bu defter 5 sınıflandırma checkpoint'ini Release'ten indirir, Food-101 val/test üzerinde yeniden değerlendirir (`src.eval_cls`), `results.csv`'ye ROC-AUC'lu yeni satırlar ve `results/probs/` altına olasılıkları yazar, GitHub'a gönderir. Eğitim yok; **~15 dk**.

In [ ]:
# 1) KURULUM — her oturumda ilk bu hücre çalışır
import os, subprocess
from kaggle_secrets import UserSecretsClient

REPO = "https://github.com/traldo44/kalori-tahmini.git"
GIT_NAME, GIT_EMAIL = "Mert Aydıngüneş", "aydingunesmert@gmail.com"   # ← başkası çalıştırırsa değiştirsin
OWNER = "mert"                                          # ← results.csv'deki owner sütunu

os.environ["GH_TOKEN"] = UserSecretsClient().get_secret("GH_TOKEN")
# Token'ı hiçbir dosyaya yazmadan git'e verir; yalnızca bu oturumun hafızasında durur.
subprocess.run(["git", "config", "--global", "credential.helper",
                '!f() { echo username=x-access-token; echo "password=$GH_TOKEN"; }; f'], check=True)

os.chdir("/kaggle/working")
if os.path.isdir("kalori-tahmini"):
    subprocess.run(["git", "-C", "kalori-tahmini", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/kaggle/working/kalori-tahmini")
subprocess.run(["git", "config", "user.name", GIT_NAME], check=True)
subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)

os.environ["KALORI_DATA"] = "/tmp/kalori-data"   # veri çıktı klasörüne (/kaggle/working) girmesin
os.environ["KALORI_OWNER"] = OWNER
print("Repo hazır →", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 2) VERİ — yalnızca Food-101 25 sınıflık alt küme (662 MB)
!python scripts/get_food101.py fetch --out /tmp/kalori-data

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU yok → Settings → Accelerator → GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 3) Checkpoint'ler Release'ten (toplam ~1 GB) ve yeniden değerlendirme
MODELS = ["resnet50_cls", "efficientnet_b0_cls", "convnext_tiny_cls", "vit_b16_cls", "vit_b16_384_cls"]
for m in MODELS:
    !python scripts/share_checkpoint.py download {m}.pt
    !python -m src.eval_cls --ckpt checkpoints/{m}.pt --name {m} || true
    !rm -f checkpoints/{m}.pt

In [ ]:
# 4) SONUÇLARI GÖNDER
!git add results && git commit -q -m "Sınıflandırma modelleri ROC-AUC ile yeniden değerlendirildi (results/probs)" && git pull --rebase -q && git push -q && echo "GitHub'a gönderildi ✓"
!grep -E "cls,.*,test," results/results.csv | tail -5 | cut -d, -f4,7,10,14